## Imports

In [21]:
import numpy
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import gym

## Network module

In [22]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

## Defining Hyperperameters


In [23]:
gamma = 0.99
epsilon = 1
epsilon_min = 0.01
epsilon_decay = 0.995
learning_rate = 0.001
batch_size = 64
memory_size = 10000

episodes = 500
target_update_freq = 10

## Network initialization

In [29]:
env = gym.make("CartPole-v1")
# env = gym.make("CartPole-v1", render_mode='human')

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

memory = deque(maxlen=memory_size)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

policy_net = QNetwork(state_size, action_size)
target_net = QNetwork(state_size, action_size)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()

## Replay buffer


In [25]:
def replay():
    if len(memory) < batch_size:
        return
    
    mini_batch = random.sample(memory, batch_size)

    states, actions, rewards, next_states, dones = zip(*mini_batch)

    states = torch.FloatTensor(states).to(device)
    actions = torch.LongTensor(actions).unsqueeze(1).to(device)
    rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
    next_states = torch.FloatTensor(next_states).to(device)
    dones = torch.FloatTensor(dones).unsqueeze(1).to(device)

    current_q = policy_net(states).gather(1, actions)

    next_q = target_net(next_states).max(1)[0].detach().unsqueeze(1)
    target_q = rewards + (gamma * next_q * (1-dones))

    loss = loss_fn(current_q, target_q)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


## Action decider
based on epsilon

In [26]:
def get_action(state, epsilon):
    if random.random() < epsilon:
        return random.choice(range(action_size))
    else:
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = policy_net(state)
        return q_values.argmax().item()

## Training Loop


In [30]:
for episode in range(episodes):
    state = env.reset()[0]
    total_reward = 0

    # Changing pole length
    env.unwrapped.length = random.uniform(0.4, 1.8)

    for t in range(500): # max steps
        action = get_action(state, epsilon)
        next_state, reward, terminated, truncated, _  = env.step(action)
        done = terminated or truncated

        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward

        replay()
        
        if done:
            break

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    if episode % target_update_freq == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if episode % 10 == 0:
        print(f"Episode: {episode}, Total Reward: {total_reward}, Epsilon: {epsilon:.3f}")

Episode: 0, Total Reward: 17.0, Epsilon: 0.806
Episode: 10, Total Reward: 28.0, Epsilon: 0.767
Episode: 20, Total Reward: 36.0, Epsilon: 0.729
Episode: 30, Total Reward: 28.0, Epsilon: 0.694
Episode: 40, Total Reward: 95.0, Epsilon: 0.660
Episode: 50, Total Reward: 56.0, Epsilon: 0.627
Episode: 60, Total Reward: 63.0, Epsilon: 0.597
Episode: 70, Total Reward: 57.0, Epsilon: 0.568
Episode: 80, Total Reward: 91.0, Epsilon: 0.540
Episode: 90, Total Reward: 119.0, Epsilon: 0.513
Episode: 100, Total Reward: 190.0, Epsilon: 0.488
Episode: 110, Total Reward: 175.0, Epsilon: 0.464
Episode: 120, Total Reward: 62.0, Epsilon: 0.442
Episode: 130, Total Reward: 212.0, Epsilon: 0.420
Episode: 140, Total Reward: 172.0, Epsilon: 0.400
Episode: 150, Total Reward: 196.0, Epsilon: 0.380
Episode: 160, Total Reward: 145.0, Epsilon: 0.361
Episode: 170, Total Reward: 149.0, Epsilon: 0.344
Episode: 180, Total Reward: 155.0, Epsilon: 0.327
Episode: 190, Total Reward: 49.0, Epsilon: 0.311
Episode: 200, Total Re